In [ ]:
## import libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, uniform
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.model_selection import train_test_split

In [ ]:
## function

In [ ]:
def modify_dataset(dataset,columns_binary):
    for i in dataset:
        print(i.dtypes)
        print(i.info())
        print(i.shape)
        i['age_60_and_above'] = i['age_60_and_above'].map({
            'Yes': 1.0,
            'No': 0.0
        })
        i['corona_result'] = i['corona_result'].map({
            'positive': 1.0,
            'negative': 0.0
        })
        i['test_indication'] = i['test_indication'].map({
            'Contact with confirmed': 'Contact',
            'Other': 'Other',
            'Abroad' : 'Abroad'
        })

        for col in columns_binary:
            i[col] = i[col].astype(float)

    explore_dataset(dataset)
    return

def explore_dataset(dataset):
    for i in dataset:
        print(i.dtypes)
        print(i.info())
        print(i.shape)

        for col in i.columns:
            print(f"Column: {col}")
            print(i[col].unique())
            print(i[col].isna().sum())
            print(i[col].value_counts(normalize=True, dropna=False))
            print("-" * 50)
    return
    
def read_data():
    data = pd.read_csv(r'C:\Users\Loic\Documents\python\Hakakton\corona_tested_individuals_ver_006.english.csv')
    data = data.dropna()
    data = data[data['corona_result'] != 'other']

    data['test_date'] = pd.to_datetime(data['test_date'])
    data = data.set_index('test_date')
    data = data.sort_index()
    return data
    
def split_paper(data):
    train_val = data.loc['2020-03-22':'2020-03-31'].copy()
    test = data.loc['2020-04-01':'2020-04-07'].copy()
    week3 = data.loc['2020-04-08':'2020-04-14'].copy()

    dataset = [train_val, test]

    columns_binary = ['cough','fever','sore_throat','shortness_of_breath','head_ache','corona_result','age_60_and_above']
    modify_dataset(dataset,columns_binary)

    ####### Hot ENCODING for Categorical variable #######
    columns_categorical = ['gender', 'test_indication']
    encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    encoder.fit(train_val[columns_categorical])

    train_encoded = encoder.transform(train_val[columns_categorical])
    test_encoded = encoder.transform(test[columns_categorical])

    feature_names = encoder.get_feature_names_out(columns_categorical)
    train_encoded = pd.DataFrame(train_encoded, columns=feature_names, index=train_val.index)
    test_encoded = pd.DataFrame(test_encoded, columns=feature_names, index=test.index)

    X_train = pd.concat([train_val.drop(columns=columns_categorical + ['corona_result']), train_encoded], axis=1)
    X_test = pd.concat([test.drop(columns=columns_categorical + ['corona_result']), test_encoded], axis=1)

    explore_dataset([X_train, X_test])

    #### target variable ##############
    y_train = train_val['corona_result']
    y_test = test['corona_result']

    return X_train, X_test, y_train, y_test

def XGBOOST_MODEL(X_train, X_test, y_train, y_test):

    neg, pos = np.bincount(y_train)
    scale_pos_weight = neg / pos
    xg = XGBClassifier(random_state=50, n_jobs=-1, objective='binary:logistic', eval_metric='logloss',
                       scale_pos_weight=scale_pos_weight)

    hyperparams = {
        'n_estimators': randint(50, 300),
        'max_depth': randint(3, 6),
        'learning_rate': uniform(0.05, 0.3),
        'min_child_weight': randint(1, 8),
        'gamma': uniform(0, 2),
        'subsample': uniform(0.7,0.3),
        'colsample_bytree': uniform(0.7, 0.3),
        'reg_alpha': uniform(0, 0.5),
        'reg_lambda': uniform(0.5, 1.5)
    }

    my_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=50)

    randomsearch = RandomizedSearchCV(
        estimator=xg,
        param_distributions=hyperparams,
        scoring='roc_auc',
        n_iter=50,
        cv=my_cv,
        refit=True,
        random_state=50,
        n_jobs=-1,
        verbose=3)

    randomsearch.fit(X_train, y_train)
    best_xg = randomsearch.best_estimator_
    best_params = randomsearch.best_params_
    print(best_params)
    return

def split_tradi(data):
    columns_binary = ['cough', 'fever', 'sore_throat', 'shortness_of_breath', 'head_ache', 'corona_result',
                      'age_60_and_above']

    dataset = [data]
    modify_dataset(dataset, columns_binary)

    X= data.drop('corona_result', axis =1).copy()
    y = data['corona_result']

    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

    ####### Hot ENCODING for Categorical variable #######
    columns_categorical = ['gender', 'test_indication']
    encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    encoder.fit(X_train[columns_categorical])

    train_encoded = encoder.transform(X_train[columns_categorical])
    test_encoded = encoder.transform(X_test[columns_categorical])

    feature_names = encoder.get_feature_names_out(columns_categorical)
    train_encoded = pd.DataFrame(train_encoded, columns=feature_names, index=X_train.index)
    test_encoded = pd.DataFrame(test_encoded, columns=feature_names, index=X_test.index)

    X_train = pd.concat([X_train.drop(columns=columns_categorical), train_encoded], axis=1)
    X_test = pd.concat([X_test.drop(columns=columns_categorical), test_encoded], axis=1)

    explore_dataset([X_train, X_test])

    return X_train, X_test, y_train, y_test

def best_param_XGBOOST(X_train, X_test, y_train, y_test, best_params, target_recall):
    neg, pos = np.bincount(y_train)
    scale_pos_weight = neg / pos
    best_xg = XGBClassifier(random_state=50, n_jobs=-1, objective='binary:logistic', eval_metric='logloss',
                       scale_pos_weight=scale_pos_weight,  **best_params)
    best_xg.fit(X_train, y_train)

    y_train_pred = best_xg.predict(X_train)
    y_test_pred = best_xg.predict(X_test)

    print('Test set')
    print(classification_report(y_test, y_test_pred, zero_division=0))
    print('Training set')
    print(classification_report(y_train, y_train_pred, zero_division=0))

    y_proba = best_xg.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()

    plt.figure()
    cm = confusion_matrix(y_test, y_test_pred, normalize='true')
    print(cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues")
    plt.title("Confusion Matrix - XGBoost 0.5 threshold")

    precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)

    print(recall)
    valid_idx = np.where(recall[:-1] >= target_recall)[0]
    best_idx = valid_idx[np.argmax(precision[:-1][valid_idx])]
    best_threshold = thresholds[best_idx]
    y_pred_opt = (y_proba >= best_threshold).astype(int)
    print("Best threshold for recall ≥ 0.90:", best_threshold)

    plt.figure()
    plt.plot(recall, precision, label=f'AP = {ap:.3f}')
    threshold_list = [0.25, 0.5, 0.75, best_threshold]

    for t in threshold_list:
        idx = np.argmin(np.abs(thresholds - t))  # closest threshold

        plt.scatter(recall[idx], precision[idx])
        plt.text(
            recall[idx],
            precision[idx],
            f"t={t:.2f}",
            fontsize=9
        )

    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()

    plt.figure()
    print(classification_report(y_test, y_pred_opt))
    cm = confusion_matrix(y_test, y_pred_opt, normalize='true')
    print(cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap="Blues")
    plt.show()
    return

In [ ]:
## run 

In [ ]:
data = read_data()
X_train, X_test, y_train, y_test = split_paper(data)
#X_train, X_test, y_train, y_test = split_tradi(data)
#XGBOOST_MODEL(X_train, X_test, y_train, y_test)
best_params = {'colsample_bytree': np.float64(0.7841768836890392), 'gamma': np.float64(1.2007029809012664), 'learning_rate': np.float64(0.20674147634809914), 'max_depth': 5, 'min_child_weight': 4, 'n_estimators': 192, 'reg_alpha': np.float64(0.24352174033383478), 'reg_lambda': np.float64(0.6057736675625042), 'subsample': np.float64(0.8746325833496754)}
best_param_XGBOOST(X_train, X_test, y_train, y_test,best_params,0.9)